# LJ EOS production runner

Самодостаточный notebook для массового сбора EOS-данных Lennard-Jones-флюида в Google Colab. Он только запускает независимые NVT MD-точки (T, rho, seed) и сохраняет численные результаты в CSV.

Графики, профили плотности, траектории, архивы и научная обработка здесь отсутствуют. Энергии в выходных CSV сохраняются в приведённых LJ-единицах на одну частицу.


In [ ]:
# Установка и импорт зависимостей.
# В Colab перед запуском выберите Runtime -> Change runtime type -> GPU.

import os, sys, math, time, csv, subprocess, traceback
from pathlib import Path
from datetime import datetime, timezone
from statistics import mean, stdev

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openmm[cuda12]', 'numpy', 'pandas', 'tqdm'])

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import openmm as mm
import openmm.app as app
from openmm import unit

print('Python:', sys.version.split()[0])
print('OpenMM:', mm.version.version)
print('\n===== nvidia-smi =====')
try:
    smi = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
    print(smi.stdout if smi.stdout else smi.stderr)
except FileNotFoundError:
    print('nvidia-smi not found')

OPENMM_PLATFORMS = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
print('OpenMM platforms:', OPENMM_PLATFORMS)
if 'CUDA' not in OPENMM_PLATFORMS:
    raise RuntimeError('CUDA is not available in OpenMM. Stop here: this production notebook is intended for Colab GPU runs, not CPU.')


In [ ]:
# Google Drive и директория вывода.

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/VPV_LJ/data')
else:
    OUTPUT_DIR = Path('/content/VPV_LJ/data')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
# Конфигурация production-run.

N = 2048
TEMPERATURES = [0.70, 0.80, 0.90, 1.00, 1.10, 1.20, 1.30]
DENSITIES = [0.0025, 0.005, 0.010, 0.020, 0.040, 0.060, 0.080, 0.100, 0.120, 0.160, 0.200, 0.240]
SEEDS = [1, 2, 3, 4, 5]

SIGMA = 1.0
EPSILON = 1.0
MASS = 1.0
RCUT = 2.5
DT = 0.002
FRICTION = 1.0

EQUIL_STEPS = 50_000
PROD_STEPS = 50_000
SAMPLE_INTERVAL = 500
N_BLOCKS = 10

DEVICE_INDEX = '0'
PRECISION = 'mixed'

print('Total points:', len(TEMPERATURES) * len(DENSITIES) * len(SEEDS))
print('Points per seed:', len(TEMPERATURES) * len(DENSITIES))


In [ ]:
# Минимальные функции production-runner.

EOS_FIELDS = ['run_id','seed','run_seed','T_target','rho_target','N','box_length','sigma','epsilon','mass','rcut','dt','friction','equil_steps','prod_steps','sample_interval','n_samples','n_blocks','platform','device_index','precision','wall_time_s','steps_per_second','T_mean','P_mean','U_mean','K_mean','E_mean','T_std','P_std','U_std','K_std','E_std','status']
BLOCK_FIELDS = ['run_id','seed','T_target','rho_target','block','step_start','step_end','n_samples','T_mean','P_mean','U_mean','K_mean','E_mean','T_std','P_std','U_std','K_std','E_std']
FAILURE_FIELDS = ['run_id','seed','run_seed','T_target','rho_target','error_type','error_message','timestamp']

def run_id_for(seed, T, rho):
    text = 'lj_seed' + str(int(seed)) + '_T' + format(float(T), '.3f') + '_rho' + format(float(rho), '.4f')
    return text.replace('.', 'p')

def deterministic_run_seed(seed, t_index, rho_index):
    return int(seed) * 1000000 + int(t_index) * 10000 + int(rho_index) * 100 + 17

def initialize_positions(N, L):
    n_side = math.ceil(N ** (1.0 / 3.0))
    spacing = L / n_side
    coords = []
    for ix in range(n_side):
        for iy in range(n_side):
            for iz in range(n_side):
                if len(coords) == N:
                    return np.asarray(coords, dtype=float)
                coords.append([(ix + 0.5) * spacing, (iy + 0.5) * spacing, (iz + 0.5) * spacing])
    return np.asarray(coords, dtype=float)

def create_lj_system(N, rho):
    L = float((int(N) / float(rho)) ** (1.0 / 3.0))
    system = mm.System()
    for _ in range(int(N)):
        system.addParticle(MASS * unit.dalton)
    system.setDefaultPeriodicBoxVectors(unit.Quantity(mm.Vec3(L,0,0), unit.nanometer), unit.Quantity(mm.Vec3(0,L,0), unit.nanometer), unit.Quantity(mm.Vec3(0,0,L), unit.nanometer))
    expr = '4*epsilon*((sigma/r)^12-(sigma/r)^6)-4*epsilon*((sigma/rcut)^12-(sigma/rcut)^6)'
    force = mm.CustomNonbondedForce(expr)
    force.addGlobalParameter('sigma', SIGMA * unit.nanometer)
    force.addGlobalParameter('epsilon', EPSILON * unit.kilojoule_per_mole)
    force.addGlobalParameter('rcut', RCUT * unit.nanometer)
    force.setNonbondedMethod(mm.CustomNonbondedForce.CutoffPeriodic)
    force.setCutoffDistance(RCUT * unit.nanometer)
    force.setUseLongRangeCorrection(False)
    for _ in range(int(N)):
        force.addParticle([])
    system.addForce(force)
    return system, L

def make_topology(N):
    topology = app.Topology()
    chain = topology.addChain()
    residue = topology.addResidue('LJ', chain)
    element = app.Element.getByAtomicNumber(18)
    for _ in range(int(N)):
        topology.addAtom('Ar', element, residue)
    return topology

def get_positions(simulation):
    state = simulation.context.getState(getPositions=True)
    return np.asarray(state.getPositions(asNumpy=True).value_in_unit(unit.nanometer), dtype=float)

def lj_virial_pressure(positions, L, T):
    V = L ** 3
    rho = len(positions) / V
    virial = 0.0
    rcut2 = RCUT * RCUT
    for i in range(len(positions) - 1):
        delta = positions[i + 1:] - positions[i]
        delta -= L * np.rint(delta / L)
        r2 = np.sum(delta * delta, axis=1)
        mask = (r2 > 0.0) & (r2 < rcut2)
        if not np.any(mask):
            continue
        inv_r2 = (SIGMA * SIGMA) / r2[mask]
        inv_r6 = inv_r2 ** 3
        inv_r12 = inv_r6 ** 2
        virial += float(np.sum(24.0 * EPSILON * (2.0 * inv_r12 - inv_r6)))
    return float(rho * T + virial / (3.0 * V))

def sample_stats(values):
    values = [float(v) for v in values if math.isfinite(float(v))]
    if not values:
        return math.nan, math.nan
    if len(values) == 1:
        return float(values[0]), 0.0
    return float(mean(values)), float(stdev(values))

def append_row_csv(path, fieldnames, row):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not path.exists() or path.stat().st_size == 0
    with path.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow({key: row.get(key, '') for key in fieldnames})

def completed_run_ids(eos_path):
    path = Path(eos_path)
    if not path.exists() or path.stat().st_size == 0:
        return set()
    df = pd.read_csv(path)
    if 'status' in df.columns:
        df = df[df['status'] == 'ok']
    return set(df['run_id'].astype(str))

def save_failure(path, run_id, seed, run_seed, T, rho, exc):
    row = {'run_id': run_id, 'seed': int(seed), 'run_seed': int(run_seed), 'T_target': float(T), 'rho_target': float(rho), 'error_type': type(exc).__name__, 'error_message': str(exc), 'timestamp': datetime.now(timezone.utc).isoformat(timespec='seconds')}
    append_row_csv(path, FAILURE_FIELDS, row)

def create_cuda_platform():
    platform = mm.Platform.getPlatformByName('CUDA')
    properties = {'DeviceIndex': str(DEVICE_INDEX), 'Precision': str(PRECISION)}
    return platform, properties

def openmm_temperature(T):
    gas_r = unit.MOLAR_GAS_CONSTANT_R.value_in_unit(unit.kilojoule_per_mole / unit.kelvin)
    return (float(T) * EPSILON / gas_r) * unit.kelvin

def run_one_point(seed, t_index, rho_index, T, rho):
    run_id = run_id_for(seed, T, rho)
    run_seed = deterministic_run_seed(seed, t_index, rho_index)
    system, L = create_lj_system(N, rho)
    target_T = openmm_temperature(T)
    integrator = mm.LangevinMiddleIntegrator(target_T, FRICTION / unit.picosecond, DT * unit.picoseconds)
    integrator.setRandomNumberSeed(int(run_seed))
    platform, properties = create_cuda_platform()
    simulation = app.Simulation(make_topology(N), system, integrator, platform, properties)
    simulation.context.setPositions(initialize_positions(N, L) * unit.nanometer)
    simulation.context.setVelocitiesToTemperature(target_T, int(run_seed))
    actual_platform = simulation.context.getPlatform().getName()
    if actual_platform != 'CUDA':
        raise RuntimeError('Expected CUDA platform, got ' + repr(actual_platform))
    start_time = time.perf_counter()
    if EQUIL_STEPS:
        simulation.step(int(EQUIL_STEPS))
    if PROD_STEPS % SAMPLE_INTERVAL != 0:
        raise ValueError('PROD_STEPS must be divisible by SAMPLE_INTERVAL')
    n_samples_expected = PROD_STEPS // SAMPLE_INTERVAL
    if n_samples_expected % N_BLOCKS != 0:
        raise ValueError('PROD_STEPS / SAMPLE_INTERVAL must be divisible by N_BLOCKS')
    samples_per_block = n_samples_expected // N_BLOCKS
    samples = []
    completed = 0
    while completed < PROD_STEPS:
        chunk = min(SAMPLE_INTERVAL, PROD_STEPS - completed)
        simulation.step(int(chunk))
        completed += int(chunk)
        state = simulation.context.getState(getEnergy=True)
        K_total = state.getKineticEnergy().value_in_unit(unit.kilojoule_per_mole)
        U_total = state.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
        T_inst = 2.0 * K_total / (max(1, 3 * N - 3) * EPSILON)
        positions = get_positions(simulation)
        P_inst = lj_virial_pressure(positions, L, T_inst)
        U_per_particle = float(U_total) / N
        K_per_particle = float(K_total) / N
        samples.append({'step': int(completed), 'T': float(T_inst), 'P': float(P_inst), 'U': U_per_particle, 'K': K_per_particle, 'E': U_per_particle + K_per_particle})
        if not math.isfinite(T_inst) or T_inst > 10.0 * float(T):
            raise RuntimeError('Unstable temperature: T_inst=' + str(T_inst))
    wall_time_s = time.perf_counter() - start_time
    steps_per_second = (EQUIL_STEPS + PROD_STEPS) / wall_time_s if wall_time_s > 0 else math.nan
    block_rows = []
    for block in range(N_BLOCKS):
        part = samples[block * samples_per_block:(block + 1) * samples_per_block]
        row = {'run_id': run_id, 'seed': int(seed), 'T_target': float(T), 'rho_target': float(rho), 'block': int(block), 'step_start': int(part[0]['step'] - SAMPLE_INTERVAL + 1), 'step_end': int(part[-1]['step']), 'n_samples': int(len(part))}
        for key in ['T', 'P', 'U', 'K', 'E']:
            avg, std = sample_stats([x[key] for x in part])
            row[key + '_mean'] = avg
            row[key + '_std'] = std
        block_rows.append(row)
    result = {'run_id': run_id, 'seed': int(seed), 'run_seed': int(run_seed), 'T_target': float(T), 'rho_target': float(rho), 'N': int(N), 'box_length': float(L), 'sigma': float(SIGMA), 'epsilon': float(EPSILON), 'mass': float(MASS), 'rcut': float(RCUT), 'dt': float(DT), 'friction': float(FRICTION), 'equil_steps': int(EQUIL_STEPS), 'prod_steps': int(PROD_STEPS), 'sample_interval': int(SAMPLE_INTERVAL), 'n_samples': int(len(samples)), 'n_blocks': int(N_BLOCKS), 'platform': actual_platform, 'device_index': str(DEVICE_INDEX), 'precision': str(PRECISION), 'wall_time_s': float(wall_time_s), 'steps_per_second': float(steps_per_second), 'status': 'ok'}
    for key in ['T', 'P', 'U', 'K', 'E']:
        avg, std = sample_stats([x[key] for x in samples])
        result[key + '_mean'] = avg
        result[key + '_std'] = std
    return result, block_rows

def cuda_smoke_check():
    old_N = globals()['N']
    try:
        globals()['N'] = 8
        result, _blocks = run_one_point(seed=0, t_index=0, rho_index=0, T=1.0, rho=0.05)
        print('CUDA smoke check ok:', result['platform'], 'steps/s:', format(result['steps_per_second'], '.1f'))
    finally:
        globals()['N'] = old_N

cuda_smoke_check()


In [ ]:
# Основной последовательный цикл по seed, температуре и плотности.

all_created_files = []
total_ok = 0
total_failures = 0
total_start = time.perf_counter()
points_per_seed = len(TEMPERATURES) * len(DENSITIES)

for seed in SEEDS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    seed_start = time.perf_counter()
    eos_path = OUTPUT_DIR / ('eos_seed_' + str(seed) + '.csv')
    blocks_path = OUTPUT_DIR / ('eos_blocks_seed_' + str(seed) + '.csv')
    failures_path = OUTPUT_DIR / ('failures_seed_' + str(seed) + '.csv')
    all_created_files.extend([eos_path, blocks_path, failures_path])
    done = completed_run_ids(eos_path)
    seed_ok_start = len(done)
    seed_errors = 0
    print('\n=== seed', seed, '===')
    print('EOS file:', eos_path)
    print('Already completed:', len(done), 'of', points_per_seed)
    point_number = 0
    progress = tqdm(total=points_per_seed, desc='seed ' + str(seed))
    for t_index, T in enumerate(TEMPERATURES):
        for rho_index, rho in enumerate(DENSITIES):
            point_number += 1
            run_id = run_id_for(seed, T, rho)
            run_seed = deterministic_run_seed(seed, t_index, rho_index)
            if run_id in done:
                progress.update(1)
                continue
            try:
                point_start = time.perf_counter()
                result, block_rows = run_one_point(seed, t_index, rho_index, T, rho)
                for block_row in block_rows:
                    append_row_csv(blocks_path, BLOCK_FIELDS, block_row)
                append_row_csv(eos_path, EOS_FIELDS, result)
                done.add(run_id)
                total_ok += 1
                point_time = time.perf_counter() - point_start
                print('seed=' + str(seed) + ' point=' + str(point_number) + '/' + str(points_per_seed) + ' T=' + format(float(T), '.2f') + ' rho=' + format(float(rho), '.4f') + ' time=' + format(point_time, '.1f') + 's speed=' + format(result['steps_per_second'], '.1f') + ' steps/s file=' + str(eos_path))
            except Exception as exc:
                seed_errors += 1
                total_failures += 1
                save_failure(failures_path, run_id, seed, run_seed, T, rho, exc)
                print('FAILED seed=' + str(seed) + ' point=' + str(point_number) + '/' + str(points_per_seed) + ' T=' + format(float(T), '.2f') + ' rho=' + format(float(rho), '.4f') + ': ' + type(exc).__name__ + ': ' + str(exc))
                traceback.print_exc(limit=2)
            progress.update(1)
    progress.close()
    seed_time = time.perf_counter() - seed_start
    seed_ok = len(completed_run_ids(eos_path))
    print('\nseed', seed, 'finished')
    print('successful points in seed file:', seed_ok)
    print('new successful points this seed:', max(0, seed_ok - seed_ok_start))
    print('errors this seed:', seed_errors)
    print('seed wall time, s:', format(seed_time, '.1f'))
    print('files:')
    print(' ', eos_path)
    print(' ', blocks_path)
    print(' ', failures_path)

total_time = time.perf_counter() - total_start
print('\nAll requested seeds finished.')
print('new successful points this run:', total_ok)
print('failures this run:', total_failures)
print('total wall time, s:', format(total_time, '.1f'))


In [ ]:
# Финальный список файлов.

print('OUTPUT_DIR:', OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.glob('*.csv')):
    try:
        rows = max(0, sum(1 for _ in path.open('r', encoding='utf-8')) - 1)
    except Exception:
        rows = '?'
    print(path.name + ': ' + str(rows) + ' data rows')

successful_total = 0
for seed in SEEDS:
    eos_path = OUTPUT_DIR / ('eos_seed_' + str(seed) + '.csv')
    successful_total += len(completed_run_ids(eos_path))
print('successful EOS points in selected seed files:', successful_total)
